In [1]:
from pathlib import Path
import os, re, random
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm import tqdm
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.home() / "Desktop/Concordia/Winter 2026/COMP 472/Project"

# Raw dataset root
RAW_ROOT = PROJECT_ROOT / "raw_data/five-car-models-images"

# Output root
OUT_ROOT = PROJECT_ROOT / "processed_data/five_car_models"

# Use the SAME class_map from Stanford Cars
CLASS_MAP_PATH = PROJECT_ROOT / "processed_data/stanford_cars/class_map.csv"

IMG_SIZE = 224
SEED = 42

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

def is_image_file(p: Path) -> bool:
    return p.suffix.lower() in IMG_EXTS

def pil_open_rgb(path: Path) -> Image.Image:
    # open + apply EXIF rotation + force RGB (fixes 1-channel & RGBA)
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im)
        im = im.convert("RGB")
        return im

def resize_with_padding(im: Image.Image, size: int) -> Image.Image:
    # keep aspect ratio, pad to square, then resize to size x size
    w, h = im.size
    if w == 0 or h == 0:
        raise ValueError("Invalid image with zero dimension")

    scale = size / max(w, h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    im = im.resize((new_w, new_h), Image.BICUBIC)

    # pad to square
    pad_w = size - new_w
    pad_h = size - new_h
    left = pad_w // 2
    top = pad_h // 2
    right = pad_w - left
    bottom = pad_h - top

    im = ImageOps.expand(im, border=(left, top, right, bottom), fill=(0, 0, 0))
    return im

def safe_filename(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "unknown"

set_seed(SEED)

# Create output folders
OUT_ROOT.mkdir(parents=True, exist_ok=True)
IMAGES_OUT = OUT_ROOT / "images_processed"
IMAGES_OUT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_ROOT:", RAW_ROOT, "| exists:", RAW_ROOT.exists())
print("OUT_ROOT:", OUT_ROOT)
print("CLASS_MAP_PATH:", CLASS_MAP_PATH, "| exists:", CLASS_MAP_PATH.exists())

PROJECT_ROOT: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project
RAW_ROOT: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/raw_data/five-car-models-images | exists: True
OUT_ROOT: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/five_car_models
CLASS_MAP_PATH: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/stanford_cars/class_map.csv | exists: True


Load SAME class map as Stanford Cars

In [2]:
assert CLASS_MAP_PATH.exists(), f"Missing class map: {CLASS_MAP_PATH}"

class_map_df = pd.read_csv(CLASS_MAP_PATH)

# Expect columns like: brand, brand_id
expected_cols = {"brand", "brand_id"}
missing = expected_cols - set(class_map_df.columns)
assert not missing, f"class_map.csv must contain columns {expected_cols}, missing: {missing}"

# Normalize brand names for matching
class_map_df["brand_norm"] = class_map_df["brand"].astype(str).str.strip().str.lower()

brand_to_id = dict(zip(class_map_df["brand_norm"], class_map_df["brand_id"]))

print("Loaded class map rows:", len(class_map_df))
print("Example brands:", class_map_df["brand"].head(10).tolist())

Loaded class map rows: 49
Example brands: ['acura', 'am general', 'aston martin', 'audi', 'bentley', 'bmw', 'bugatti', 'buick', 'cadillac', 'chevrolet']


Build file list from five-car-models-images

In [3]:
# the brand was drived from folder name (prefix before _models_images).

assert RAW_ROOT.exists(), f"RAW_ROOT not found: {RAW_ROOT}"

top_dirs = sorted([p for p in RAW_ROOT.iterdir() if p.is_dir()])
assert len(top_dirs) > 0, f"No brand folders found in: {RAW_ROOT}"

print("Found brand folders:")
for d in top_dirs:
    print(" -", d.name)

# Map folder prefix -> standardized brand name used in the Stanford class_map
FOLDER_BRAND_TO_CLASSMAP_BRAND = {
    "audi": "audi",
    "bentley": "bentley",
    "bmw": "bmw",
    # common Stanford naming is often "mercedes-benz"
    "mercedes": "mercedes-benz",
    "toyota": "toyota",
}

rows = []
for brand_dir in top_dirs:
    prefix = brand_dir.name.split("_", 1)[0].strip().lower()
    mapped_brand = FOLDER_BRAND_TO_CLASSMAP_BRAND.get(prefix, prefix)

    imgs = [p for p in brand_dir.rglob("*") if p.is_file() and is_image_file(p)]
    for p in imgs:
        rows.append({"img_path": str(p), "brand": mapped_brand})

df_raw = pd.DataFrame(rows)
print("\nTotal raw images found:", len(df_raw))
print("Brands found:", sorted(df_raw["brand"].unique().tolist()))
df_raw.head()

Found brand folders:
 - audi_models_images
 - bentley_models_images
 - bmw_models_images
 - mercedes_models_images
 - toyota_models_images

Total raw images found: 1646
Brands found: ['audi', 'bentley', 'bmw', 'mercedes-benz', 'toyota']


,img_path,brand
0,/Users/pegahaghili/Desktop/Concordia/Winter 20...,audi
1,/Users/pegahaghili/Desktop/Concordia/Winter 20...,audi
2,/Users/pegahaghili/Desktop/Concordia/Winter 20...,audi
3,/Users/pegahaghili/Desktop/Concordia/Winter 20...,audi
4,/Users/pegahaghili/Desktop/Concordia/Winter 20...,audi


Attach brand_id using existing class_map

In [4]:
df_raw["brand_norm"] = df_raw["brand"].astype(str).str.strip().str.lower()

missing_brands = sorted(set(df_raw["brand_norm"]) - set(brand_to_id.keys()))
if missing_brands:
    raise ValueError(
        "Some brands in this dataset are not present in your Stanford class_map.csv.\n"
        f"Missing brands: {missing_brands}\n\n"
        "Fix by either:\n"
        " 1) updating FOLDER_BRAND_TO_CLASSMAP_BRAND to match the exact brand names in class_map.csv, or\n"
        " 2) adding these brands to your class_map.csv (not recommended if you want to keep the exact same ids)."
    )

df_raw["brand_id"] = df_raw["brand_norm"].map(brand_to_id).astype(int)

# keep only what we need
df = df_raw[["img_path", "brand", "brand_id"]].copy()
print("Ready rows:", len(df))
df.head()

Ready rows: 1646


,img_path,brand,brand_id
0,/Users/pegahaghili/Desktop/Concordia/Winter 20...,audi,3
1,/Users/pegahaghili/Desktop/Concordia/Winter 20...,audi,3
2,/Users/pegahaghili/Desktop/Concordia/Winter 20...,audi,3
3,/Users/pegahaghili/Desktop/Concordia/Winter 20...,audi,3
4,/Users/pegahaghili/Desktop/Concordia/Winter 20...,audi,3


Cleaning / Preprocessing

In [5]:
bad_files = []
records = []

for r in tqdm(df.itertuples(index=False), total=len(df), desc="Cleaning Five Car Models"):
    src = Path(r.img_path)
    if not src.exists():
        bad_files.append({"file": str(src), "error": "File not found"})
        continue

    try:
        im = pil_open_rgb(src)
        im = resize_with_padding(im, IMG_SIZE)

        # brand folder under images_processed/
        brand_folder = safe_filename(r.brand)
        out_dir = IMAGES_OUT / brand_folder
        out_dir.mkdir(parents=True, exist_ok=True)

        out_path = out_dir / (src.stem + ".jpg")
        im.save(out_path, format="JPEG", quality=95, optimize=True)

        rel_path = str(out_path.relative_to(PROJECT_ROOT)).replace("\\", "/")

        records.append({
            "path": rel_path,
            "brand": r.brand,
            "brand_id": int(r.brand_id),
        })

    except Exception as e:
        bad_files.append({"file": str(src), "error": str(e)})

manifest = pd.DataFrame(records)
bad_df = pd.DataFrame(bad_files)

print("Processed:", len(manifest))
print("Bad files:", len(bad_df))

# Save manifest + bad files
manifest_path = OUT_ROOT / "manifest.csv"
manifest.to_csv(manifest_path, index=False)

if len(bad_df) > 0:
    bad_df.to_csv(OUT_ROOT / "bad_files.csv", index=False)

print("Saved manifest:", manifest_path)
manifest.head()

Cleaning Five Car Models:  45%|████▍     | 738/1646 [00:09<00:10, 87.50it/s]/Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/.venv/lib/python3.14/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Cleaning Five Car Models: 100%|██████████| 1646/1646 [00:19<00:00, 85.41it/s]


Processed: 1644
Bad files: 2
Saved manifest: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/five_car_models/manifest.csv


,path,brand,brand_id
0,processed_data/five_car_models/images_processe...,audi,3
1,processed_data/five_car_models/images_processe...,audi,3
2,processed_data/five_car_models/images_processe...,audi,3
3,processed_data/five_car_models/images_processe...,audi,3
4,processed_data/five_car_models/images_processe...,audi,3


Save CSVs

In [6]:
# Save class_map.csv for this dataset which is a subset of the one for Stanford
# only for the brands present here.

class_map_out = (
    manifest[["brand", "brand_id"]]
    .drop_duplicates()
    .sort_values("brand_id")
    .reset_index(drop=True)
)

class_map_out_path = OUT_ROOT / "class_map.csv"
class_map_out.to_csv(class_map_out_path, index=False)
print("Saved class map:", class_map_out_path)
class_map_out

Saved class map: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/five_car_models/class_map.csv


,brand,brand_id
0,audi,3
1,bentley,4
2,bmw,5
3,mercedes-benz,33
4,toyota,46


Stats

In [7]:
# brand_counts.csv + stats_full.txt
brand_counts = (
    manifest.groupby(["brand_id", "brand"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

brand_counts_path = OUT_ROOT / "brand_counts.csv"
brand_counts.to_csv(brand_counts_path, index=False)
print("Saved:", brand_counts_path)

stats_lines = [
    "Five Car Models (Kaggle) - Preprocessing Stats",
    f"Total processed images: {len(manifest)}",
    f"Unique brands: {manifest['brand'].nunique()}",
    f"Target image size: {IMG_SIZE}x{IMG_SIZE}",
    f"Bad/corrupt/missing images removed: {len(bad_df)}",
    f"Class map used: {CLASS_MAP_PATH}",
    f"Class map subset saved at: {class_map_out_path}",
]

(OUT_ROOT / "stats_full.txt").write_text("\n".join(stats_lines), encoding="utf-8")
print("\n".join(stats_lines))

brand_counts

Saved: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/five_car_models/brand_counts.csv
Five Car Models (Kaggle) - Preprocessing Stats
Total processed images: 1644
Unique brands: 5
Target image size: 224x224
Bad/corrupt/missing images removed: 2
Class map used: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/stanford_cars/class_map.csv
Class map subset saved at: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/five_car_models/class_map.csv


,brand_id,brand,count
2,5,bmw,618
3,33,mercedes-benz,459
0,3,audi,260
4,46,toyota,227
1,4,bentley,80


Splitting the data train/validation/test (80/10/10)


In [8]:
# Split into train/val/test and save manifests
manifest_full = pd.read_csv(OUT_ROOT / "manifest.csv")

print("Total images:", len(manifest_full))
print("Unique brands:", manifest_full["brand"].nunique())

# Train (80%) vs Temp (20%)
train_df, temp_df = train_test_split(
    manifest_full,
    test_size=0.20,
    stratify=manifest_full["brand"],
    random_state=SEED
)

# Val (10%) vs Test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["brand"],
    random_state=SEED
)

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

train_path = OUT_ROOT / "manifest_train.csv"
val_path   = OUT_ROOT / "manifest_val.csv"
test_path  = OUT_ROOT / "manifest_test.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("Saved:")
print(" -", train_path)
print(" -", val_path)
print(" -", test_path)

# Quick distribution check
print("\nTrain distribution:\n", train_df["brand"].value_counts())
print("\nVal distribution:\n", val_df["brand"].value_counts())
print("\nTest distribution:\n", test_df["brand"].value_counts())

Total images: 1644
Unique brands: 5
Train size: 1315
Val size: 164
Test size: 165
Saved:
 - /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/five_car_models/manifest_train.csv
 - /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/five_car_models/manifest_val.csv
 - /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/five_car_models/manifest_test.csv

Train distribution:
 brand
bmw              494
mercedes-benz    367
audi             208
toyota           182
bentley           64
Name: count, dtype: int64

Val distribution:
 brand
bmw              62
mercedes-benz    46
audi             26
toyota           22
bentley           8
Name: count, dtype: int64

Test distribution:
 brand
bmw              62
mercedes-benz    46
audi             26
toyota           23
bentley           8
Name: count, dtype: int64


Sample images

In [9]:
SAMPLES_ROOT = PROJECT_ROOT / "processed_data/samples/five_car_models"
SAMPLES_ROOT.mkdir(parents=True, exist_ok=True)

samples_per_brand = 3

for brand, group in manifest.groupby("brand"):
    brand_safe = safe_filename(brand)
    out_dir = SAMPLES_ROOT / brand_safe
    out_dir.mkdir(parents=True, exist_ok=True)

    # Randomly sample up to 3 images per brand
    sample_rows = group.sample(n=min(samples_per_brand, len(group)), random_state=SEED)

    for i, r in enumerate(sample_rows.itertuples(index=False), start=1):
        src = PROJECT_ROOT / r.path
        dst = out_dir / f"sample_{i}.jpg"
        dst.write_bytes(src.read_bytes())

print("Sample images created at:", SAMPLES_ROOT)

Sample images created at: /Users/pegahaghili/Desktop/Concordia/Winter 2026/COMP 472/Project/processed_data/samples/five_car_models
